# 06b build hagrid - build the HAGRID retrieval data

**Source:** HAGRID (`miracl/hagrid`)
**Produces:** `sampled_questions.json` + `chunks.json` -- the same two files
`datasets/nq/01_build_nq.ipynb` produces for KILT-NQ, so `02x` -> `03` -> `04x` -> `05`
run on this dataset unchanged.

**Before running:** Settings -> Internet: **ON**. Accelerator: **None/CPU** -- there is no
model in this notebook.

Only Stage 2 (LOAD) is specific to HAGRID. Stages 3-7 are emitted verbatim into
all three builder notebooks by `pipeline/scripts/generate_builder_notebooks.py`. Edit that
script, not this file.

Upload the output as Kaggle Dataset `mehedikhan72/rag-faith-hagrid-retrieval` when the run finishes.

## Stage 0 - Install pinned dependencies

`datasets>=4.0` dropped loading-script datasets entirely, and `miracl/hagrid` /
`miracl/miracl-corpus` are still script-based -- the same reason
`01_build_nq.ipynb` pins an older version for `facebook/kilt_tasks`. Pinning here too
keeps every builder on one `datasets` version, so a schema difference between datasets is
never a library-version artifact.

**Restart the kernel after this cell if Kaggle's preinstalled version conflicts.**
CPU is enough for the whole notebook -- there is no model here. Save the GPU quota for 02x.

In [ ]:
!pip install -q "datasets==2.19.0" "fsspec==2023.9.2" rank_bm25
print("Installed. If this is the first install this session, restart the kernel now.")

## Stage 1 - Config

`N_QUESTIONS` and `SEED` match the NQ run exactly. `CHUNK_SIZE_TOKENS = 100` is Wallat et
al.'s chunk size and must not drift between datasets -- the adversarial test plants a
4-word span into a chunk, so chunk length is part of the treatment.

In [ ]:
DATASET_KEY = "hagrid"

import hashlib
import json
import random
import re
import statistics
from pathlib import Path

# ---- Identical across all builders (and to 01_build_nq.ipynb) ----
N_QUESTIONS = 500
N_DISTRACTORS = 5000
CHUNK_SIZE_TOKENS = 100
SEED = 42
# -------------------------------------------------------------------

OUT_DIR = Path("/kaggle/working/retrieval_pipeline_data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)


def write_json(path, obj, indent=None):
    with open(path, "w") as f:
        json.dump(obj, f, indent=indent)


def doc_id(title, text):
    """Stable, content-addressed document id.

    NQ has real `wikipedia_id`s; QAMPARI passages carry no guaranteed id and RAGTruth
    passages carry none at all. Hashing title+text gives an id that is stable across runs
    and machines, and that deduplicates identical passages reached from different
    questions -- which is what the corpus needs.
    """
    return hashlib.sha1((title + "\n" + text).encode("utf-8")).hexdigest()[:12]


print(f"dataset: {DATASET_KEY}")
print(f"N_QUESTIONS={N_QUESTIONS}  N_DISTRACTORS={N_DISTRACTORS}  SEED={SEED}")
print(f"output: {OUT_DIR}")

## Stage 2 - LOAD: HAGRID

Train (1,922) and dev (716) are pooled to 2,638 eligible queries. Both splits carry the
same human attributability annotations and we regenerate answers ourselves, so the split
boundary carries no information here -- pooling only widens the draw.

**`quotes` has no guaranteed `title` field**, so it is probed on the first record rather
than assumed; when absent, `title` is empty and `text == body_text` downstream.

**Distractors come from `miracl/miracl-corpus`** (English, the corpus HAGRID's quotes were
drawn from). Note honestly what the sampling is: a streaming shuffle draws from a buffer
and a shuffled shard order, *not* uniformly over all ~32.9M passages. That is fine for
distractors, whose only job is to be topically unrelated -- but the write-up must not
describe it as a uniform sample of MIRACL.

In [ ]:
from datasets import load_dataset


ds = load_dataset("miracl/hagrid", trust_remote_code=True)
print({k: len(v) for k, v in ds.items()})

first_quote = next(q for split in ds.values() for r in split
                   for q in (r.get("quotes") or []))
print(f"quote fields: {sorted(first_quote.keys())}")
HAS_TITLE = "title" in first_quote
print(f"quotes carry a title field: {HAS_TITLE}")

pool = []
for split_name, split in ds.items():
    for r in split:
        quotes = r.get("quotes") or []
        if not quotes:
            continue
        answers = r.get("answers") or []
        passages = [{"title": (q.get("title") or "") if HAS_TITLE else "",
                     "text": q.get("text", "") or ""} for q in quotes]
        passages = [p for p in passages if p["text"].strip()]
        if not passages:
            continue
        pool.append({
            "id": f"{r['query_id']}",
            "question": r["query"],
            "answer": (answers[0].get("answer", "") if answers else ""),
            "passages": passages,
            # Every quote is human-labelled relevant, so all of them are gold.
            "gold_passage_idxs": list(range(len(passages))),
        })

print(f"pool: {len(pool)} queries with quotes")
print(f"mean quotes/query: {sum(len(p['passages']) for p in pool) / len(pool):.2f}")

corpus_stream = load_dataset("miracl/miracl-corpus", "en", split="train",
                             streaming=True, trust_remote_code=True)

# miracl-corpus passages DO carry a title; HAGRID's quotes do not. Keeping the title on
# distractors only would render every distractor as "Title: text" and every gold passage
# as bare text -- a visible cue in the prompt that separates gold from distractor without
# reading either. The model could then cite on formatting rather than content, which is
# precisely the behaviour this study measures. Titles are dropped so the whole corpus is
# uniformly untitled, matching the quotes.
distractors = [{"title": "", "text": d.get("text", "") or ""}
               for d in corpus_stream.shuffle(seed=SEED, buffer_size=10_000)
                                     .take(N_DISTRACTORS)]
distractors = [d for d in distractors if d["text"].strip()]
print(f"distractors sampled from miracl-corpus/en: {len(distractors)} (titles dropped)")

## Stage 3 - Sample 500 questions, uniformly at random

`random.sample` without replacement from the eligible pool, seeded -- the same draw
`01_build_nq.ipynb` makes. No stratification and no filtering after the draw: the sample
is meant to represent its own dataset, not to be bent into matching NQ's shape.

Stage 3b then *checks* that the draw is representative rather than assuming it.

In [ ]:
assert len(pool) >= N_QUESTIONS, (
    f"pool has only {len(pool)} eligible questions, need {N_QUESTIONS}"
)

sample_idx = random.sample(range(len(pool)), N_QUESTIONS)
sample = [pool[i] for i in sample_idx]

print(f"Pool: {len(pool)} eligible questions -> sampled {len(sample)}")
print(f"Example: {sample[0]['question']!r}")
print(f"  passages: {len(sample[0]['passages'])}, gold idxs: {sample[0]['gold_passage_idxs']}")

## Stage 3b - Distribution check: is the sample representative of the pool?

A uniform draw is *expected* to be representative; this cell produces the evidence that it
is, so the claim is checkable rather than asserted. Three covariates -- question length,
answer length, passages per question -- summarised for pool and sample, plus a two-sample
Kolmogorov-Smirnov statistic.

**This is a report, not a filter.** Nothing is re-drawn on a large KS statistic. Re-drawing
until a test passes would bias the sample, which is exactly what the uniform draw avoids.
A large D here is something to disclose in the write-up, not to sample away.

In [ ]:
from scipy import stats

def covariates(rows):
    return {
        "question_len_words": [len(r["question"].split()) for r in rows],
        "answer_len_words": [len(r["answer"].split()) for r in rows],
        "n_passages": [len(r["passages"]) for r in rows],
    }


def describe(xs):
    return {
        "n": len(xs),
        "mean": round(statistics.mean(xs), 3),
        "sd": round(statistics.pstdev(xs), 3),
        "median": statistics.median(xs),
        "min": min(xs),
        "max": max(xs),
    }


pool_cov, sample_cov = covariates(pool), covariates(sample)
dist_report = {"dataset": DATASET_KEY, "seed": SEED,
               "n_pool": len(pool), "n_sample": len(sample), "covariates": {}}

for name in pool_cov:
    d, p = stats.ks_2samp(pool_cov[name], sample_cov[name])
    dist_report["covariates"][name] = {
        "pool": describe(pool_cov[name]),
        "sample": describe(sample_cov[name]),
        "ks_statistic": round(float(d), 4),
        "ks_pvalue": round(float(p), 4),
    }

write_json(OUT_DIR / "sample_distribution_report.json", dist_report, indent=2)

print(f"{'covariate':<20} {'pool mean':>10} {'sample mean':>12} {'KS D':>8} {'p':>8}")
for name, r in dist_report["covariates"].items():
    print(f"{name:<20} {r['pool']['mean']:>10.2f} {r['sample']['mean']:>12.2f} "
          f"{r['ks_statistic']:>8.4f} {r['ks_pvalue']:>8.4f}")
print("\n(p > 0.05 => no detectable difference between sample and pool)")

## Stage 4 - Assemble the corpus

The corpus is the sampled questions' own passages, deduplicated by content hash, plus
whatever distractors the LOAD stage produced. Distractors are load-bearing, not filler: a
corpus containing only each question's own gold passages makes retrieval trivial, and a
faithfulness test on trivial retrieval measures nothing.

In [ ]:
corpus = {}


def add_doc(title, text):
    did = doc_id(title, text)
    if did not in corpus:
        corpus[did] = {"wikipedia_id": did, "title": title, "text": text}
    return did


questions = []
for r in sample:
    passage_ids = [add_doc(p["title"], p["text"]) for p in r["passages"]]
    gold = sorted({passage_ids[i] for i in r["gold_passage_idxs"]
                   if 0 <= i < len(passage_ids)})
    questions.append({
        "id": str(r["id"]),
        "question": r["question"],
        "answer": r["answer"],
        "gold_wikipedia_ids": gold,
    })

n_from_questions = len(corpus)
for d in distractors:
    add_doc(d["title"], d["text"])

n_gold_empty = sum(1 for q in questions if not q["gold_wikipedia_ids"])
print(f"{n_from_questions} unique docs from the sampled questions")
print(f"+ {len(corpus) - n_from_questions} distractors = {len(corpus)} documents")
print(f"questions with no gold document: {n_gold_empty}/{len(questions)}")

write_json(OUT_DIR / "sampled_questions.json", questions, indent=2)
write_json(OUT_DIR / "corpus.json", list(corpus.values()))

## Stage 5 - Chunk into ~100-token passages

Two text fields per chunk, exactly as `01_build_nq.ipynb` writes them:

- `text` = `"{title}: {piece}"` -- what the model is shown at generation time
- `body_text` = `piece` alone -- what BM25 indexes

Splitting them is the retrieval-diversity fix from 2026-08-08: indexing title+body gave
every chunk of a document the same title-term boost, so top-5 filled up with chunks of one
document. HAGRID and RAGTruth passages can have no title at all, in which case the two
fields are equal and the fix is a no-op for them.

In [ ]:
chunks = []
for doc in corpus.values():
    words = doc["text"].split()
    for i in range(0, len(words), CHUNK_SIZE_TOKENS):
        piece = " ".join(words[i:i + CHUNK_SIZE_TOKENS])
        if not piece.strip():
            continue
        title = doc["title"]
        chunks.append({
            "chunk_id": f"{doc['wikipedia_id']}_{i // CHUNK_SIZE_TOKENS}",
            "wikipedia_id": doc["wikipedia_id"],
            "title": title,
            "text": f"{title}: {piece}" if title else piece,
            "body_text": piece,
        })

write_json(OUT_DIR / "chunks.json", chunks)
n_untitled = sum(1 for c in chunks if not c["title"])
print(f"Chunked {len(corpus)} documents into {len(chunks)} passages "
      f"of ~{CHUNK_SIZE_TOKENS} tokens.")
print(f"chunks with no title (text == body_text): {n_untitled}")

# Titling must be all-or-nothing across the corpus. If gold passages were untitled and
# distractors titled (or vice versa), the "{title}: " prefix would mark which passages are
# relevant without reading them -- a cue the model could cite on instead of content.
gold_ids = {g for q in questions for g in q["gold_wikipedia_ids"]}
gold_untitled = [c for c in chunks if c["wikipedia_id"] in gold_ids and not c["title"]]
n_gold_chunks = sum(1 for c in chunks if c["wikipedia_id"] in gold_ids)
assert n_untitled in (0, len(chunks)), (
    f"mixed titling: {n_untitled}/{len(chunks)} chunks untitled, of which "
    f"{len(gold_untitled)}/{n_gold_chunks} are gold -- titles must be uniform across "
    f"the corpus or they leak which passages are relevant"
)
print("titling is uniform across the corpus")

## Stage 6 - BM25 index + retrieval diversity diagnostic

Same diagnostic `01_build_nq.ipynb` Stage 6b runs, over all 500 questions. Two numbers
decide whether this dataset is worth spending GPU on:

- **gold-hit@5** -- does retrieval surface a gold document at all? NQ reached 95.4%.
- **mean unique documents in top-5** -- NQ reached 2.05 after the body-text fix. If top-5
  collapses to a single document, the `cited_other` adversarial variant has nothing to
  work with.

A dataset far outside those is a retrieval problem to fix *before* generation, not after.

In [ ]:
from collections import Counter

from rank_bm25 import BM25Okapi

_TOKEN_RE = re.compile(r"\w+")


def tokenize(text):
    return _TOKEN_RE.findall(text.lower())


bm25 = BM25Okapi([tokenize(c["body_text"]) for c in chunks])
print(f"BM25 index built over {len(chunks)} chunks (indexed on body_text).")


def retrieve_top_k(query, k=5):
    scores = bm25.get_scores(tokenize(query))
    top = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [chunks[i] for i in top]


unique_counts, gold_hits, single_doc = [], 0, []
gold_evaluable = 0
for q in questions:
    top5 = retrieve_top_k(q["question"], k=5)
    doc_ids = [c["wikipedia_id"] for c in top5]
    unique_counts.append(len(set(doc_ids)))
    if q["gold_wikipedia_ids"]:
        gold_evaluable += 1
        if set(q["gold_wikipedia_ids"]) & set(doc_ids):
            gold_hits += 1
    if len(set(doc_ids)) == 1:
        single_doc.append({"id": q["id"], "doc_id": doc_ids[0],
                           "is_gold": doc_ids[0] in q["gold_wikipedia_ids"]})

n = len(questions)
report = {
    "dataset": DATASET_KEY,
    "n_questions": n,
    "n_chunks": len(chunks),
    "n_documents": len(corpus),
    "unique_doc_count_distribution": {str(k): v for k, v in
                                      sorted(Counter(unique_counts).items())},
    "mean_unique_docs_in_top5": round(sum(unique_counts) / n, 3),
    # Denominator is questions that HAVE a gold label; QAMPARI can legitimately have none.
    "gold_evaluable_questions": gold_evaluable,
    "gold_hit_at_5": gold_hits,
    "gold_hit_at_5_pct": round(100 * gold_hits / max(gold_evaluable, 1), 2),
    "n_single_document_top5": len(single_doc),
    "n_single_document_top5_pct": round(100 * len(single_doc) / n, 2),
    "single_document_cases": single_doc,
}
write_json(OUT_DIR / "retrieval_diversity_report.json", report, indent=2)

print(f"gold-hit@5             : {gold_hits}/{gold_evaluable} "
      f"({report['gold_hit_at_5_pct']}%)      [NQ: 95.4%]")
print(f"mean unique docs top-5 : {report['mean_unique_docs_in_top5']}"
      f"                  [NQ: 2.05]")
print(f"single-document top-5  : {report['n_single_document_top5_pct']}%"
      f"                     [NQ: 40%]")
print(f"distribution           : {report['unique_doc_count_distribution']}")

## Stage 7 - Validate against the contract, then persist

The assertions below are the same ones the pipeline will rely on downstream. Failing here,
in the notebook that produced the data, is much cheaper than failing in a GPU kernel.

In [ ]:
import shutil

q = json.loads((OUT_DIR / "sampled_questions.json").read_text())
c = json.loads((OUT_DIR / "chunks.json").read_text())

assert len(q) == N_QUESTIONS, f"expected {N_QUESTIONS} questions, got {len(q)}"
assert {k for r in q for k in r} == {"id", "question", "answer", "gold_wikipedia_ids"}
assert {k for r in c for k in r} == {"chunk_id", "wikipedia_id", "title", "text", "body_text"}
assert len({r["id"] for r in q}) == len(q), "duplicate question ids"
assert len({r["chunk_id"] for r in c}) == len(c), "duplicate chunk ids"
assert all(r["question"].strip() for r in q), "empty question text"
assert all(r["body_text"].strip() for r in c), "empty chunk body"

known = {r["wikipedia_id"] for r in c}
dangling = {g for r in q for g in r["gold_wikipedia_ids"] if g not in known}
assert not dangling, f"{len(dangling)} gold ids missing from chunks.json"

shutil.make_archive("/kaggle/working/retrieval_pipeline_data", "zip", OUT_DIR)
print("CONTRACT OK")
print(f"  {len(q)} questions, {len(c)} chunks, {len(corpus)} documents")
for f in sorted(OUT_DIR.iterdir()):
    print(f"  {f.name:38} {f.stat().st_size / 1e6:8.2f} MB")
print("\nZipped to /kaggle/working/retrieval_pipeline_data.zip")